# Convolutional Neural Network (CNN)

Notebook này xây dựng một mô hình **Convolutional Neural Network** từ đầu bằng NumPy để phân loại ảnh nhỏ.

Mục tiêu chính:

1. Hiểu cách lớp **convolution** trích xuất đặc trưng không gian từ ảnh.
2. Hiểu vai trò của **ReLU**, **Max Pooling**, **Flatten**, **Dense**, và **Softmax**.
3. Cài đặt forward pass và backward pass cơ bản.
4. Huấn luyện CNN trên bộ dữ liệu ảnh tự tạo gồm hai lớp: vạch dọc và vạch ngang.

Mô hình được viết thủ công để dễ học nguyên lý. Với bài toán thực tế lớn hơn, bạn thường dùng PyTorch, TensorFlow hoặc Keras.

## 1. Chuẩn bị thư viện

Ta dùng:

- `numpy` để tính toán ma trận và tensor.
- `matplotlib` để hiển thị ảnh, loss và accuracy.

Nếu môi trường chưa có thư viện, cài bằng:

```bash
pip install numpy matplotlib
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## 2. Tạo bộ dữ liệu ảnh đơn giản

Ta tạo ảnh kích thước `8 x 8` gồm hai loại:

- Lớp `0`: ảnh có một vạch **dọc**.
- Lớp `1`: ảnh có một vạch **ngang**.

Mỗi ảnh được thêm nhiễu nhỏ để mô hình không chỉ ghi nhớ mẫu cố định. Dữ liệu có shape:

```text
X: (số ảnh, số kênh, chiều cao, chiều rộng)
y: (số ảnh,)
```

Vì ảnh xám nên số kênh bằng `1`.

In [ ]:
def make_line_dataset(n_samples=300, image_size=8, noise=0.18):
    X = np.zeros((n_samples, 1, image_size, image_size), dtype=float)
    y = np.zeros(n_samples, dtype=int)

    for i in range(n_samples):
        label = np.random.randint(0, 2)
        y[i] = label

        image = np.zeros((image_size, image_size), dtype=float)
        position = np.random.randint(1, image_size - 1)

        if label == 0:
            image[:, position] = 1.0
        else:
            image[position, :] = 1.0

        image += np.random.normal(0, noise, size=image.shape)
        image = np.clip(image, 0, 1)
        X[i, 0] = image

    return X, y


X, y = make_line_dataset()

split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)
print('X_test shape:', X_test.shape)
print('y_test shape:', y_test.shape)

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(8, 3))

for ax, image, label in zip(axes.ravel(), X[:12, 0], y[:12]):
    ax.imshow(image, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'label={label}')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 3. Kiến trúc CNN

Mô hình có kiến trúc:

```text
Input image 8x8x1
        |
        v
Conv2D: 4 filters, kernel 3x3
        |
        v
ReLU
        |
        v
MaxPool 2x2
        |
        v
Flatten
        |
        v
Dense layer
        |
        v
Softmax output: 2 classes
```

Giải thích nhanh:

- **Conv2D** dùng filter nhỏ quét qua ảnh để tìm pattern cục bộ như cạnh, vạch, góc.
- **ReLU** giữ giá trị dương và đưa giá trị âm về `0`, giúp mô hình học phi tuyến.
- **MaxPool** giảm kích thước feature map và giữ đặc trưng nổi bật nhất trong từng vùng.
- **Flatten** biến tensor thành vector.
- **Dense + Softmax** tạo xác suất cho từng lớp.

## 4. Cài đặt các layer

Ta viết thủ công forward pass và backward pass cho các thành phần chính.

Trong code dưới đây:

- `conv2d_forward`: tính convolution hợp lệ, không padding.
- `conv2d_backward`: tính gradient cho filter và bias.
- `relu_forward`, `relu_backward`: lan truyền qua hàm kích hoạt ReLU.
- `maxpool_forward`, `maxpool_backward`: lấy max trong từng vùng `2 x 2`.
- `softmax` và `cross_entropy`: chuyển logits thành xác suất và tính loss.

In [ ]:
def conv2d_forward(X, filters, bias):
    n_samples, n_channels, height, width = X.shape
    n_filters, _, kernel_h, kernel_w = filters.shape
    out_h = height - kernel_h + 1
    out_w = width - kernel_w + 1
    output = np.zeros((n_samples, n_filters, out_h, out_w))

    for n in range(n_samples):
        for f in range(n_filters):
            for i in range(out_h):
                for j in range(out_w):
                    region = X[n, :, i:i + kernel_h, j:j + kernel_w]
                    output[n, f, i, j] = np.sum(region * filters[f]) + bias[f]

    return output


def conv2d_backward(X, filters, d_output):
    n_samples, _, height, width = X.shape
    n_filters, _, kernel_h, kernel_w = filters.shape
    _, _, out_h, out_w = d_output.shape

    d_filters = np.zeros_like(filters)
    d_bias = np.zeros(n_filters)
    d_X = np.zeros_like(X)

    for n in range(n_samples):
        for f in range(n_filters):
            d_bias[f] += np.sum(d_output[n, f])
            for i in range(out_h):
                for j in range(out_w):
                    region = X[n, :, i:i + kernel_h, j:j + kernel_w]
                    d_filters[f] += region * d_output[n, f, i, j]
                    d_X[n, :, i:i + kernel_h, j:j + kernel_w] += filters[f] * d_output[n, f, i, j]

    return d_X, d_filters, d_bias


def relu_forward(X):
    return np.maximum(0, X)


def relu_backward(d_output, X):
    return d_output * (X > 0)


def maxpool_forward(X, pool_size=2, stride=2):
    n_samples, n_channels, height, width = X.shape
    out_h = (height - pool_size) // stride + 1
    out_w = (width - pool_size) // stride + 1
    output = np.zeros((n_samples, n_channels, out_h, out_w))

    for n in range(n_samples):
        for c in range(n_channels):
            for i in range(out_h):
                for j in range(out_w):
                    h_start = i * stride
                    w_start = j * stride
                    region = X[n, c, h_start:h_start + pool_size, w_start:w_start + pool_size]
                    output[n, c, i, j] = np.max(region)

    return output


def maxpool_backward(d_output, X, pool_size=2, stride=2):
    d_X = np.zeros_like(X)
    n_samples, n_channels, out_h, out_w = d_output.shape

    for n in range(n_samples):
        for c in range(n_channels):
            for i in range(out_h):
                for j in range(out_w):
                    h_start = i * stride
                    w_start = j * stride
                    region = X[n, c, h_start:h_start + pool_size, w_start:w_start + pool_size]
                    mask = region == np.max(region)
                    d_X[n, c, h_start:h_start + pool_size, w_start:w_start + pool_size] += mask * d_output[n, c, i, j]

    return d_X


def softmax(logits):
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values, axis=1, keepdims=True)


def cross_entropy(probs, y_true, eps=1e-8):
    n_samples = len(y_true)
    correct_probs = probs[np.arange(n_samples), y_true]
    return -np.mean(np.log(correct_probs + eps))

## 5. Forward pass của CNN

Forward pass là quá trình đưa input đi qua từng layer để tạo dự đoán.

Dòng dữ liệu:

```text
X -> Conv2D -> ReLU -> MaxPool -> Flatten -> Dense -> Softmax
```

Ta lưu các giá trị trung gian trong `cache` để dùng lại khi tính gradient ở backward pass.

In [ ]:
def initialize_parameters():
    params = {}
    params['filters'] = np.random.normal(0, 0.1, size=(4, 1, 3, 3))
    params['conv_bias'] = np.zeros(4)
    params['dense_w'] = np.random.normal(0, 0.1, size=(4 * 3 * 3, 2))
    params['dense_b'] = np.zeros(2)
    return params


def forward(X, params):
    conv = conv2d_forward(X, params['filters'], params['conv_bias'])
    relu = relu_forward(conv)
    pool = maxpool_forward(relu)
    flat = pool.reshape(X.shape[0], -1)
    logits = flat @ params['dense_w'] + params['dense_b']
    probs = softmax(logits)

    cache = {
        'conv': conv,
        'relu': relu,
        'pool': pool,
        'flat': flat,
        'logits': logits,
        'probs': probs,
    }

    return probs, cache


def predict(X, params):
    probs, _ = forward(X, params)
    return np.argmax(probs, axis=1)


def accuracy(X, y, params):
    y_pred = predict(X, params)
    return np.mean(y_pred == y)

## 6. Backward pass và cập nhật tham số

Backward pass tính gradient từ loss quay ngược về các layer trước.

Với softmax + cross-entropy, gradient theo logits có dạng đơn giản:

$$\frac{\partial L}{\partial z} = \frac{p - y}{N}$$

Trong đó:

- `p` là vector xác suất dự đoán.
- `y` là one-hot label.
- `N` là batch size.

Sau đó gradient đi ngược qua Dense, Flatten, MaxPool, ReLU và Conv2D.

In [ ]:
def one_hot(y, n_classes=2):
    encoded = np.zeros((len(y), n_classes))
    encoded[np.arange(len(y)), y] = 1
    return encoded


def backward(X, y, params, cache):
    n_samples = X.shape[0]
    y_encoded = one_hot(y)

    d_logits = (cache['probs'] - y_encoded) / n_samples

    d_dense_w = cache['flat'].T @ d_logits
    d_dense_b = np.sum(d_logits, axis=0)

    d_flat = d_logits @ params['dense_w'].T
    d_pool = d_flat.reshape(cache['pool'].shape)
    d_relu = maxpool_backward(d_pool, cache['relu'])
    d_conv = relu_backward(d_relu, cache['conv'])
    _, d_filters, d_conv_bias = conv2d_backward(X, params['filters'], d_conv)

    grads = {
        'filters': d_filters,
        'conv_bias': d_conv_bias,
        'dense_w': d_dense_w,
        'dense_b': d_dense_b,
    }

    return grads


def update_parameters(params, grads, learning_rate):
    for key in params:
        params[key] -= learning_rate * grads[key]
    return params

## 7. Huấn luyện mô hình

Ta huấn luyện bằng mini-batch gradient descent:

1. Chia dữ liệu thành các batch nhỏ.
2. Chạy forward pass để lấy xác suất dự đoán.
3. Tính cross-entropy loss.
4. Chạy backward pass để lấy gradient.
5. Cập nhật filter convolution và trọng số dense layer.

Do dữ liệu nhỏ và mô hình đơn giản, loss thường giảm rất nhanh.

In [ ]:
def iterate_minibatches(X, y, batch_size=32):
    indices = np.random.permutation(len(X))

    for start in range(0, len(X), batch_size):
        batch_indices = indices[start:start + batch_size]
        yield X[batch_indices], y[batch_indices]


params = initialize_parameters()
epochs = 25
batch_size = 32
learning_rate = 0.25
history = {'loss': [], 'train_acc': [], 'test_acc': []}

for epoch in range(1, epochs + 1):
    batch_losses = []

    for X_batch, y_batch in iterate_minibatches(X_train, y_train, batch_size):
        probs, cache = forward(X_batch, params)
        batch_loss = cross_entropy(probs, y_batch)
        grads = backward(X_batch, y_batch, params, cache)
        params = update_parameters(params, grads, learning_rate)
        batch_losses.append(batch_loss)

    train_acc = accuracy(X_train, y_train, params)
    test_acc = accuracy(X_test, y_test, params)
    mean_loss = float(np.mean(batch_losses))

    history['loss'].append(mean_loss)
    history['train_acc'].append(train_acc)
    history['test_acc'].append(test_acc)

    if epoch == 1 or epoch % 5 == 0:
        print(
            f'Epoch {epoch:02d} | loss = {mean_loss:.4f} | '
            f'train_acc = {train_acc:.3f} | test_acc = {test_acc:.3f}'
        )

## 8. Đánh giá kết quả

Ta quan sát loss và accuracy qua từng epoch. Nếu mô hình học tốt, loss giảm dần và accuracy tiến gần `1.0`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))

axes[0].plot(history['loss'], linewidth=2)
axes[0].set_title('Training loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-entropy')
axes[0].grid(alpha=0.25)

axes[1].plot(history['train_acc'], label='train', linewidth=2)
axes[1].plot(history['test_acc'], label='test', linewidth=2)
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1.05)
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

print(f'Final train accuracy: {accuracy(X_train, y_train, params):.3f}')
print(f'Final test accuracy: {accuracy(X_test, y_test, params):.3f}')

## 9. Xem thử dự đoán

Các ảnh bên dưới lấy từ tập test. Tiêu đề hiển thị:

```text
true label -> predicted label
```

Với bài toán này:

- `0`: vạch dọc.
- `1`: vạch ngang.

In [ ]:
sample_count = 12
sample_images = X_test[:sample_count]
sample_labels = y_test[:sample_count]
sample_preds = predict(sample_images, params)

fig, axes = plt.subplots(2, 6, figsize=(8, 3))

for ax, image, true_label, pred_label in zip(
    axes.ravel(), sample_images[:, 0], sample_labels, sample_preds
):
    ax.imshow(image, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'{true_label} -> {pred_label}')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 10. Trực quan hóa filter đã học

Filter convolution là các ma trận nhỏ `3 x 3`. Sau khi huấn luyện, mỗi filter có thể phản ứng mạnh với một kiểu pattern cục bộ nào đó trong ảnh.

Với dữ liệu vạch dọc/ngang, các filter thường học được các cấu trúc gần giống detector cho đường thẳng, cạnh hoặc vùng sáng.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(7, 2))

for i, ax in enumerate(axes):
    ax.imshow(params['filters'][i, 0], cmap='coolwarm')
    ax.set_title(f'filter {i}')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 11. Tóm tắt mô hình

Mạng CNN trong notebook này có pipeline:

```text
Ảnh đầu vào
   |
   v
Convolution: trích xuất đặc trưng cục bộ
   |
   v
ReLU: thêm phi tuyến
   |
   v
MaxPool: giảm kích thước và giữ đặc trưng mạnh
   |
   v
Flatten: chuyển feature map thành vector
   |
   v
Dense + Softmax: phân loại
```

Điểm cần nhớ:

- **CNN phù hợp với ảnh** vì nó giữ cấu trúc không gian của dữ liệu.
- **Filter convolution** học các pattern nhỏ và dùng lại cùng filter trên toàn ảnh.
- **Pooling** giúp giảm số chiều và làm mô hình bền hơn với dịch chuyển nhỏ.
- **Dense layer** dùng các đặc trưng đã trích xuất để quyết định lớp cuối cùng.
- Với dữ liệu thật, nên dùng framework như PyTorch/Keras để huấn luyện nhanh, ổn định và dễ mở rộng hơn.